# 10a — Data Loading, QC & PCA

**Goal:** Load pseudobulk counts from `12_fragment_matrices/`, merge across species,
apply quality filters, and run PCA on **all shared regions** (pre-filtering) and
**filtered shared peaks** (post-filtering), plus **per-cell-type subsets**.

**Outputs saved to** `cross_species_consensus_v3/13_deseq2_R_pseudobulk/`:
- `pseudobulk/pb_data_shared.rds` — filtered shared-peak counts + metadata
- `pseudobulk/pb_data_union.rds`  — union-peak counts + metadata (for evolutionary branch testing)
- `plots/*All_Regions_PCA_*.pdf`  — PCA on all shared regions (before peak filtering)
- `plots/*Filtered_Peaks_PCA_*.pdf` — PCA on filtered shared peaks
- `plots/*_PCA_*.pdf`             — Per-cell-type PCA
- `plots/Count_Distributions_Per_Species_Log10.pdf`

**Input:** Parquet files from `cross_species_consensus_v3/12_fragment_matrices/{Species}/`

In [1]:
# =============================================================================
# Cell 1: Load Packages & Source Utility Functions
# =============================================================================
suppressPackageStartupMessages({
  library(DESeq2)
  library(arrow)
  library(ggplot2)
  library(pheatmap)
  library(dplyr)
  library(tibble)
  library(tidyr)
  library(readr)
})

# Source all reusable pipeline functions
source("../src/deseq2_utils.R")
message("All packages loaded & utility functions sourced.")

All packages loaded & utility functions sourced.



In [2]:
# =============================================================================
# Cell 2: Configuration
# =============================================================================
BASE      <- "/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
QUANT_DIR <- file.path(BASE, "cross_species_consensus_v3/12_fragment_matrices")
OUT_DIR   <- file.path(BASE, "cross_species_consensus_v3/13_deseq2_R_pseudobulk")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

SPECIES <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")

# Filtering thresholds
MIN_SAMPLE_COUNTS <- 50000
MIN_CELLS         <- 100

# Aggregation settings
DO_AGGREGATE   <- TRUE
COLS_TO_MERGE  <- c("region")
COLS_TO_SUM    <- c("n_cells")

# Peak filtering
MIN_SAMPLES_PER_SPECIES <- 2
MIN_SPECIES_ACTIVE      <- 6
MAX_COUNT_THRESHOLD     <- 150

message("Configuration set. Output: ", OUT_DIR)

Configuration set. Output: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk



In [3]:
# =============================================================================
# Cell 3: Load & Merge Pseudobulk Data
# =============================================================================
raw <- load_pseudobulk_data(QUANT_DIR, SPECIES)

# Inner join (shared peaks) — for global PCA and shared-peak DESeq2
shared <- merge_pseudobulk(raw$all_counts, raw$all_meta, join_type = "inner")
counts_merged <- shared$counts
meta_merged   <- shared$meta

Loaded Human: 254 pseudobulk samples, 1039336 peaks

Loaded Bonobo: 102 pseudobulk samples, 992889 peaks

Loaded Chimpanzee: 103 pseudobulk samples, 1029053 peaks

Loaded Gorilla: 161 pseudobulk samples, 1013198 peaks

Loaded Macaque: 82 pseudobulk samples, 982905 peaks

Loaded Marmoset: 112 pseudobulk samples, 971722 peaks

Inner join: 829088 shared peaks

Merged matrix: 829088 peaks x 814 samples



In [4]:
# =============================================================================
# Cell 4: Filter to Adult Samples & Aggregate
# =============================================================================
# Subset to Adults
meta_merged   <- meta_merged[meta_merged$age == "Adult", ]
counts_merged <- counts_merged[, rownames(meta_merged)]
message("Adult samples: ", ncol(counts_merged))

# Aggregate across regions (collapse biological replicates from different regions)
if (DO_AGGREGATE && length(COLS_TO_MERGE) > 0) {
  agg <- aggregate_pseudobulk(counts_merged, meta_merged,
                              merge_cols = COLS_TO_MERGE,
                              sum_cols   = COLS_TO_SUM)
  counts_merged <- agg$counts
  meta_merged   <- agg$meta
}

head(meta_merged)

Adult samples: 814

Aggregating across: region

Grouping by: cell_type, donor, age, species

Warning message:
“There was 1 warning in `summarize()`.
ℹ In argument: `across(all_of(actual_sum_cols), sum, na.rm = TRUE)`.
ℹ In group 1: `agg_id = "Adipocytes_Almerida 17079_Adult_Marmoset"`.
Caused by warning:
! The `...` argument of `across()` is deprecated as of dplyr 1.1.0.
Supply arguments directly to `.fns` through an anonymous function instead.

  # Previously
  across(a:b, mean, na.rm = TRUE)

  # Now
  across(a:b, \(x) mean(x, na.rm = TRUE))”
Reduced from 814 to 674 samples.



,cell_type,donor,region,age,label,sample_id,species,agg_id,n_cells
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>
Adipocytes_Almerida 17079_Adult_Marmoset,Adipocytes,Almerida 17079,Merged,Adult,Adipocytes_Almerida 17079_Adult_Marmoset,Adipocytes_Almerida 17079_Adult_Marmoset,Marmoset,Adipocytes_Almerida 17079_Adult_Marmoset,42
Adipocytes_B010_Adult_Human,Adipocytes,B010,Merged,Adult,Adipocytes_B010_Adult_Human,Adipocytes_B010_Adult_Human,Human,Adipocytes_B010_Adult_Human,21
Adipocytes_B012_Adult_Human,Adipocytes,B012,Merged,Adult,Adipocytes_B012_Adult_Human,Adipocytes_B012_Adult_Human,Human,Adipocytes_B012_Adult_Human,16
Adipocytes_BN075_Adult_Chimpanzee,Adipocytes,BN075,Merged,Adult,Adipocytes_BN075_Adult_Chimpanzee,Adipocytes_BN075_Adult_Chimpanzee,Chimpanzee,Adipocytes_BN075_Adult_Chimpanzee,7
Adipocytes_BN085_Adult_Chimpanzee,Adipocytes,BN085,Merged,Adult,Adipocytes_BN085_Adult_Chimpanzee,Adipocytes_BN085_Adult_Chimpanzee,Chimpanzee,Adipocytes_BN085_Adult_Chimpanzee,17
Adipocytes_BN114_Adult_Bonobo,Adipocytes,BN114,Merged,Adult,Adipocytes_BN114_Adult_Bonobo,Adipocytes_BN114_Adult_Bonobo,Bonobo,Adipocytes_BN114_Adult_Bonobo,10


In [5]:
# =============================================================================
# Cell 5: QC — Count Distributions & Sample Filtering
# =============================================================================
meta_merged$total_counts <- colSums(counts_merged)

# Plot count distributions
plots_dir <- file.path(OUT_DIR, "plots")
dir.create(plots_dir, showWarnings = FALSE, recursive = TRUE)

plot_count_distribution(meta_merged,
                        file.path(plots_dir, "Count_Distributions_Per_Species_Log10.pdf"),
                        min_counts = MIN_SAMPLE_COUNTS,
                        min_cells  = MIN_CELLS)

# Apply quality filters
filt <- filter_samples(counts_merged, meta_merged,
                       min_counts = MIN_SAMPLE_COUNTS,
                       min_cells  = MIN_CELLS)
counts_filtered <- filt$counts
meta_filtered   <- filt$meta

# Setup factors
meta_filtered$species   <- factor(meta_filtered$species, levels = SPECIES)
meta_filtered$donor     <- factor(meta_filtered$donor)
meta_filtered$age       <- factor(meta_filtered$age)
meta_filtered$region    <- factor(meta_filtered$region)
meta_filtered$is_human  <- factor(ifelse(meta_filtered$species == "Human",
                                         "Human", "NonHuman"),
                                  levels = c("NonHuman", "Human"))
meta_filtered$cell_type <- as.factor(make.names(as.character(meta_filtered$cell_type)))

  Count distribution plot saved.

Filtering: counts >= 50000 AND cells >= 100

Kept 355 / 674 samples.



In [11]:
# =============================================================================
# Cell 7: Global PCA — All Shared Peaks
# =============================================================================
vsd_global <- run_pca_suite(counts_filtered, meta_filtered, OUT_DIR,
                            prefix = "Shared_Peaks")

message("Global PCA on all shared peaks complete.")

  PCA design: ~species + cell_type

converting counts to integer mode

using ntop=500 top features by variance

  Saved: Shared_Peaks_PCA_species.pdf

  Saved: Shared_Peaks_PCA_cell_type.pdf

  Saved: Shared_Peaks_PCA_region.pdf

  Saved: Shared_Peaks_PCA_age.pdf

  Saved: Shared_Peaks_PCA_donor.pdf

Warning message:
“Removed 265 rows containing missing values or values outside the scale range
(`geom_point()`).”
Warning message:
“ggrepel: 287 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
  Saved: Shared_Peaks_PCA_Combined.pdf

PCA suite complete for: Shared_Peaks

Global PCA on all shared peaks complete.



In [6]:
# =============================================================================
# Cell 6: Subset to Shared Cell Types & Species-Aware Peak Filtering
# =============================================================================
shared_ct <- subset_shared_celltypes(counts_filtered, meta_filtered)
counts_shared <- shared_ct$counts
meta_shared   <- shared_ct$meta

# Species-aware peak filtering
keep_peaks <- species_aware_peak_filter(
  counts_shared, meta_shared, SPECIES,
  min_samples_per_species = MIN_SAMPLES_PER_SPECIES,
  min_species_active      = MIN_SPECIES_ACTIVE,
  max_count_threshold     = MAX_COUNT_THRESHOLD
)
counts_shared <- counts_shared[keep_peaks, ]

message("Final shared-peak matrix: ", nrow(counts_shared), " peaks x ",
        ncol(counts_shared), " samples")

Shared cell types (13): Colonocytes, Crypt.Fibroblasts.WNT2B., ECs, Enterocytes, Goblet.cells, Macrophages, Naive.B.cells, Plasma.B.cells, Specialized.Fibroblasts.RSPO3..only, Specialized.Fibroblasts.SYNM., Stem.cells, T.cells, TA.cells

Kept 241 samples.

Species-aware filtering: kept 71663 / 829088 peaks (8.6%)

Final shared-peak matrix: 71663 peaks x 241 samples



In [8]:
# =============================================================================
# Cell 8: Global PCA — Filtered Shared Peaks
# =============================================================================
vsd_global <- run_pca_suite(counts_shared, meta_shared, OUT_DIR,
                            prefix = "Filtered_Peaks")

message("Global PCA on filtered shared peaks (", nrow(counts_shared),
        " peaks) complete.")

  PCA design: ~species + cell_type

converting counts to integer mode

using ntop=500 top features by variance

  Saved: Filtered_Peaks_PCA_species.pdf

  Saved: Filtered_Peaks_PCA_cell_type.pdf

  Saved: Filtered_Peaks_PCA_region.pdf

  Saved: Filtered_Peaks_PCA_age.pdf

  Saved: Filtered_Peaks_PCA_donor.pdf

Warning message:
“Removed 100 rows containing missing values or values outside the scale range
(`geom_point()`).”
Warning message:
“ggrepel: 17 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
  Saved: Filtered_Peaks_PCA_Combined.pdf

PCA suite complete for: Filtered_Peaks

Global PCA on filtered shared peaks (71663 peaks) complete.



In [9]:
# =============================================================================
# Cell 8: Per-Cell-Type PCA Subsets
# =============================================================================
# Run PCA for each cell type individually
cell_types <- levels(meta_shared$cell_type)

for (ct in cell_types) {
  meta_ct   <- meta_shared[meta_shared$cell_type == ct, ]
  counts_ct <- counts_shared[, rownames(meta_ct)]

  # Skip if too few samples
  if (ncol(counts_ct) < 4 || length(unique(meta_ct$species)) < 2) {
    message("Skipping PCA for ", ct, ": too few samples.")
    next
  }

  # Light peak filter for this specific cell type
  keep_ct   <- rowSums(counts_ct >= 10) >= 2
  counts_ct <- counts_ct[keep_ct, ]

  tryCatch({
    run_pca_suite(counts_ct, meta_ct, OUT_DIR, prefix = ct)
  }, error = function(e) {
    message("  PCA failed for ", ct, ": ", e$message)
  })
}

message("\nPer-cell-type PCA complete.")

  PCA design: ~species

converting counts to integer mode

using ntop=500 top features by variance

  Saved: Colonocytes_PCA_species.pdf

  Saved: Colonocytes_PCA_cell_type.pdf

  Saved: Colonocytes_PCA_region.pdf

  Saved: Colonocytes_PCA_age.pdf

  Saved: Colonocytes_PCA_donor.pdf

  Saved: Colonocytes_PCA_Combined.pdf

PCA suite complete for: Colonocytes

  PCA design: ~species

converting counts to integer mode

using ntop=500 top features by variance

  Saved: Crypt.Fibroblasts.WNT2B._PCA_species.pdf

  Saved: Crypt.Fibroblasts.WNT2B._PCA_cell_type.pdf

  Saved: Crypt.Fibroblasts.WNT2B._PCA_region.pdf

  Saved: Crypt.Fibroblasts.WNT2B._PCA_age.pdf

  Saved: Crypt.Fibroblasts.WNT2B._PCA_donor.pdf

  Saved: Crypt.Fibroblasts.WNT2B._PCA_Combined.pdf

PCA suite complete for: Crypt.Fibroblasts.WNT2B.

  PCA design: ~species

converting counts to integer mode

using ntop=500 top features by variance

  Saved: ECs_PCA_species.pdf

  Saved: ECs_PCA_cell_type.pdf

  Saved: ECs_PCA_region.p

In [10]:
# =============================================================================
# Cell 9: Save Intermediate Data for Downstream Notebooks
# =============================================================================
save_dir <- file.path(OUT_DIR, "pseudobulk")
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---- Shared-peak data (for 10b shared-peak analysis) ----
saveRDS(list(counts = counts_shared, meta = meta_shared),
        file.path(save_dir, "pb_data_shared.rds"))

# ---- Union-peak data (for 10b contrast-peak analysis) ----
# Build a union-join matrix, then apply the SAME aggregation / filtering
# pipeline so that the column names match meta_shared (post-aggregation IDs).
union_raw <- merge_pseudobulk(raw$all_counts, raw$all_meta, join_type = "union")

# 1. Subset to Adults (same filter as Cell 4)
union_meta   <- union_raw$meta[union_raw$meta$age == "Adult", ]
union_counts <- union_raw$counts[, rownames(union_meta)]

# 2. Aggregate (same settings as Cell 4)
if (DO_AGGREGATE && length(COLS_TO_MERGE) > 0) {
  agg_union    <- aggregate_pseudobulk(union_counts, union_meta,
                                       merge_cols = COLS_TO_MERGE,
                                       sum_cols   = COLS_TO_SUM)
  union_counts <- agg_union$counts
  union_meta   <- agg_union$meta
}

# 3. Apply same quality filter (same thresholds as Cell 5)
union_meta$total_counts <- colSums(union_counts)
filt_union   <- filter_samples(union_counts, union_meta,
                               min_counts = MIN_SAMPLE_COUNTS,
                               min_cells  = MIN_CELLS)
union_counts <- filt_union$counts
union_meta   <- filt_union$meta

# 4. Keep only the samples that survived shared-cell-type filtering
valid_samples     <- intersect(rownames(meta_shared), colnames(union_counts))
counts_union_filt <- union_counts[, valid_samples, drop = FALSE]
meta_union_filt   <- meta_shared[valid_samples, ]

message("Union matrix after full pipeline: ", nrow(counts_union_filt),
        " peaks x ", ncol(counts_union_filt), " samples")

saveRDS(list(counts = counts_union_filt, meta = meta_union_filt),
        file.path(save_dir, "pb_data_union.rds"))

# ---- Global VST for heatmaps in 10b ----
saveRDS(vsd_global, file.path(save_dir, "vsd_global_shared.rds"))

message("\n=== All intermediate data saved ===")
message("  pb_data_shared.rds  : ", nrow(counts_shared), " peaks x ",
        ncol(counts_shared), " samples")
message("  pb_data_union.rds   : ", nrow(counts_union_filt), " peaks x ",
        ncol(counts_union_filt), " samples")
message("  vsd_global_shared.rds")

Union join: 1142003 total peaks

Merged matrix: 1142003 peaks x 814 samples

Aggregating across: region

Grouping by: cell_type, donor, age, species

Reduced from 814 to 674 samples.

Filtering: counts >= 50000 AND cells >= 100

Kept 355 / 674 samples.

Union matrix after full pipeline: 1142003 peaks x 241 samples


=== All intermediate data saved ===

  pb_data_shared.rds  : 71663 peaks x 241 samples

  pb_data_union.rds   : 1142003 peaks x 241 samples

  vsd_global_shared.rds

